# Context Compression Assistant

## Retrieve Many Chunks, Compress to a Smaller Evidence Set, Then Answer

**Project 25** - Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Context compression for RAG pipelines |
| Strategies | Reranking, Extractive compression, Abstractive summary |
| Retrieval | Dense embedding search (sentence-transformers) |
| Corpus | 30-chunk cloud computing knowledge base |
| Evaluation | Answer quality, context reduction ratio, information retention |


## Project Overview

### The Problem: Too Much Context

RAG systems retrieve chunks to give the generator evidence. But:

- **Retrieving too few chunks** misses relevant information (low recall)
- **Retrieving too many chunks** dilutes signal with noise (low precision)
- **Long context confuses generators** -- they lose focus with 10+ chunks
- **Token costs scale linearly** with context length

### The Solution: Retrieve Wide, Compress Tight

```
Question --> Retrieve K=10 chunks (high recall)
                  |
         Compress to 3-5 chunks (high precision)
                  |
         Generate answer from compressed context
```

Three compression strategies:

| Strategy | Method | Output |
|----------|--------|--------|
| **Reranking** | Score each chunk's relevance, keep top-N | Subset of original chunks |
| **Extractive** | Keep only relevant sentences from each chunk | Shortened chunks |
| **Abstractive** | Summarize all chunks into one paragraph | New condensed text |


## Learning Goals

1. Understand **why** context compression is needed in RAG
2. Implement three compression strategies: reranking, extractive, abstractive
3. Measure the **compression ratio** (how much context is reduced)
4. Measure **information retention** (how much relevant info survives)
5. Compare answer quality across strategies
6. Learn when each strategy is appropriate


## Problem Statement

Given a question and K=10 retrieved chunks:

1. **Rerank** chunks by relevance and keep only top-3
2. **Extract** only the sentences relevant to the question
3. **Summarize** all chunks into a concise evidence paragraph
4. Compare the three approaches against using all 10 chunks (no compression)

Success criteria: Compression should **reduce context by 50-80%** while
**retaining 80%+ of the relevant information**.


## Why Context Compression Matters

| Without compression | With compression |
|---------------------|------------------|
| 10 chunks, ~3000 tokens | 3 chunks or ~500 tokens |
| Generator sees noise | Generator sees only evidence |
| Higher token cost | 50-80% cost reduction |
| Lost-in-the-middle effect | Focused, coherent context |
| Slower inference | Faster generation |

**Lost-in-the-middle:** Research shows LLMs struggle to use information
from the middle of long contexts. Compression puts key info front and center.


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random, textwrap
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K_RETRIEVE = 10    # retrieve many chunks (high recall)
K_RERANK   = 3     # keep top-N after reranking
RELEVANCE_THRESHOLD = 0.35  # sentence relevance for extractive compression

print(f"Config: model={EMBEDDING_MODEL}")
print(f"  Retrieve K={K_RETRIEVE}, Rerank top-{K_RERANK}")
print(f"  Extractive sentence threshold={RELEVANCE_THRESHOLD}")


## Knowledge Base

A 30-chunk corpus about cloud computing. Long enough that retrieving
K=10 chunks returns a mix of relevant and tangential content --
exactly the scenario where compression helps.


In [ ]:
corpus = [
    {"id": "C01", "text": "Cloud computing delivers computing resources over the internet on a pay-as-you-go basis. The three main service models are Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). IaaS provides virtual machines and storage. PaaS provides development platforms. SaaS provides ready-to-use applications."},
    {"id": "C02", "text": "Serverless computing automatically manages infrastructure and scales to zero when idle. AWS Lambda, Azure Functions, and Google Cloud Functions are popular serverless platforms. Serverless is event-driven and charges only for actual compute time used. Cold starts can add 100-500ms latency on first invocation."},
    {"id": "C03", "text": "Containers package applications with their dependencies into portable units. Docker is the standard container runtime. Container images are lightweight compared to virtual machines because they share the host OS kernel. Containers typically start in seconds compared to minutes for VMs."},
    {"id": "C04", "text": "Kubernetes orchestrates containers at scale. It handles deployment, scaling, load balancing, and self-healing. Key Kubernetes concepts include Pods, Services, Deployments, and ConfigMaps. Kubernetes runs on all major cloud providers and on-premises."},
    {"id": "C05", "text": "Cloud cost optimization strategies include right-sizing instances, using reserved or spot instances, implementing auto-scaling, and monitoring unused resources. Reserved instances offer 30-60% savings over on-demand pricing for predictable workloads. Spot instances offer up to 90% savings but can be interrupted."},
    {"id": "C06", "text": "Multi-cloud strategy uses services from multiple cloud providers to avoid vendor lock-in. Benefits include best-of-breed services, geographic coverage, and risk mitigation. Challenges include increased complexity, data transfer costs, and skill requirements across platforms."},
    {"id": "C07", "text": "Cloud security follows the shared responsibility model. The provider secures the infrastructure. The customer secures their data, applications, and access management. Key practices include encryption at rest and in transit, identity and access management (IAM), and regular security audits."},
    {"id": "C08", "text": "Infrastructure as Code (IaC) manages cloud resources through configuration files. Terraform, AWS CloudFormation, and Pulumi are popular IaC tools. IaC enables version control, repeatability, and automated provisioning. Drift detection helps identify manual changes that deviate from the defined state."},
    {"id": "C09", "text": "Cloud-native applications are designed specifically for cloud environments. They use microservices architecture, containerization, dynamic orchestration, and DevOps practices. The twelve-factor app methodology provides guidelines for building cloud-native applications. Key principles include stateless processes and disposable instances."},
    {"id": "C10", "text": "Cloud monitoring uses tools like Prometheus, Grafana, Datadog, and CloudWatch. Key metrics include CPU utilization, memory usage, network throughput, error rates, and latency percentiles. Alerting rules notify teams when metrics exceed thresholds. Distributed tracing tracks requests across microservices."},
    {"id": "C11", "text": "Database services in the cloud include relational (RDS, Cloud SQL), NoSQL (DynamoDB, Cosmos DB), and graph databases (Neptune, ArangoDB). Managed database services handle backups, patching, and replication. Database choice depends on data model, query patterns, and consistency requirements."},
    {"id": "C12", "text": "Content Delivery Networks (CDNs) cache content at edge locations worldwide. CloudFront, Azure CDN, and Cloudflare are major CDN providers. CDNs reduce latency by serving content from the nearest edge location. They also help absorb DDoS attacks by distributing traffic."},
    {"id": "C13", "text": "Cloud networking includes Virtual Private Clouds (VPCs), subnets, security groups, and load balancers. VPCs isolate cloud resources in a private network. Security groups act as virtual firewalls. Load balancers distribute traffic across multiple instances for high availability."},
    {"id": "C14", "text": "CI/CD pipelines automate building, testing, and deploying applications. Jenkins, GitHub Actions, GitLab CI, and Azure DevOps are popular CI/CD tools. Pipelines typically include stages for linting, unit tests, integration tests, security scans, and deployment. Blue-green and canary deployments reduce risk."},
    {"id": "C15", "text": "Cloud storage services include object storage (S3, Blob Storage), block storage (EBS, Managed Disks), and file storage (EFS, Azure Files). Object storage is ideal for unstructured data like images and logs. Block storage provides high-performance volumes for databases. Storage tiering reduces costs by moving infrequently accessed data to cheaper tiers."},
    {"id": "C16", "text": "Auto-scaling adjusts resource capacity based on demand. Horizontal scaling adds or removes instances. Vertical scaling resizes existing instances. Predictive auto-scaling uses machine learning to anticipate demand changes. Scaling policies define triggers based on metrics like CPU utilization or request queue depth."},
    {"id": "C17", "text": "Cloud compliance frameworks include SOC 2, ISO 27001, HIPAA, PCI DSS, and GDPR. Cloud providers offer compliance certifications for their infrastructure. Customers must ensure their applications and data handling meet regulatory requirements. Compliance audits verify adherence to security and privacy standards."},
    {"id": "C18", "text": "Microservices architecture decomposes applications into small, independent services. Each service has its own database and can be deployed independently. Service communication uses REST APIs, gRPC, or message queues. Benefits include independent scaling, technology diversity, and fault isolation."},
    {"id": "C19", "text": "Cloud disaster recovery strategies include backup and restore, pilot light, warm standby, and multi-site active-active. Recovery Time Objective (RTO) defines acceptable downtime. Recovery Point Objective (RPO) defines acceptable data loss. Multi-region deployments provide the highest availability but at higher cost."},
    {"id": "C20", "text": "FinOps is a financial operations framework for managing cloud costs. It combines engineering, finance, and business teams to optimize cloud spending. Key practices include cost allocation tagging, budget alerts, and regular cost reviews. Cloud cost management tools include AWS Cost Explorer, Azure Cost Management, and GCP Billing."},
    {"id": "C21", "text": "Service mesh technologies like Istio, Linkerd, and Consul Connect manage service-to-service communication. They provide traffic management, security (mTLS), and observability without changing application code. Service meshes add a sidecar proxy to each service instance that intercepts network traffic."},
    {"id": "C22", "text": "Edge computing processes data closer to where it is generated rather than in a centralized cloud data center. Use cases include IoT processing, real-time analytics, and low-latency applications. AWS Outposts, Azure Stack Edge, and Google Distributed Cloud bring cloud services to edge locations."},
    {"id": "C23", "text": "Cloud-native security practices include zero-trust networking, pod security policies, image scanning, and secrets management. Tools like Vault, Sealed Secrets, and AWS Secrets Manager handle sensitive configuration data. Runtime security monitors containers for suspicious behavior."},
    {"id": "C24", "text": "Observability consists of three pillars: logs, metrics, and traces. Structured logging with correlation IDs enables tracing requests across services. Metrics provide quantitative measurements of system behavior. Distributed traces show the full journey of a request through microservices."},
    {"id": "C25", "text": "Cloud migration strategies follow the 7 Rs: Rehost (lift-and-shift), Replatform, Refactor, Repurchase, Retire, Retain, and Relocate. Assessment involves evaluating application dependencies, data volumes, and compliance requirements. Migration waves group applications by priority and complexity."},
    {"id": "C26", "text": "API gateways manage API traffic and provide authentication, rate limiting, request transformation, and analytics. Kong, AWS API Gateway, and Apigee are popular API gateway solutions. API versioning ensures backward compatibility. GraphQL provides flexible query capabilities compared to REST."},
    {"id": "C27", "text": "Cloud-native message queues include SQS, Azure Service Bus, Google Pub/Sub, and Apache Kafka. Message queues decouple producers from consumers enabling asynchronous processing. Dead letter queues capture failed messages for troubleshooting. Event-driven architectures use events to trigger downstream processing."},
    {"id": "C28", "text": "GitOps uses Git as the single source of truth for declarative infrastructure and application deployments. Tools like ArgoCD and Flux watch Git repositories and automatically sync cluster state. Pull-based deployments are more secure than push-based because the cluster pulls changes rather than having CI push to it."},
    {"id": "C29", "text": "Cloud cost allocation uses tags and labels to attribute spending to teams, projects, or environments. Chargeback models bill teams for their actual usage. Showback models display costs without direct billing. Consistent tagging policies are essential for accurate cost reporting and optimization."},
    {"id": "C30", "text": "Chaos engineering tests system resilience by deliberately introducing failures. Netflix Chaos Monkey randomly terminates instances. AWS Fault Injection Simulator and Gremlin provide managed chaos testing. Steady-state hypothesis defines normal system behavior. Experiments verify the system maintains steady state under failure conditions."},
]

print(f"Knowledge base: {len(corpus)} chunks")
total_chars = sum(len(d["text"]) for d in corpus)
print(f"Total corpus size: {total_chars:,} characters")
print(f"Average chunk size: {total_chars // len(corpus)} characters")


## Dense Retriever

Encode all chunks. Retrieve top-K=10 for high recall.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [doc["text"] for doc in corpus]
chunk_embeddings = encoder.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)
chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)


def retrieve(query: str, k: int = K_RETRIEVE) -> List[Dict]:
    """Retrieve top-k chunks by cosine similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (chunk_embeddings @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "text": corpus[i]["text"],
             "score": float(scores[i]), "rank": r + 1}
            for r, i in enumerate(top_idx)]


# Sanity check
test = retrieve("cloud cost optimization", k=5)
print(f"\nDense retriever ready (index shape: {chunk_embeddings.shape})")
print(f"Test: top-5 for 'cloud cost optimization'")
for r in test:
    print(f"  [{r["id"]}] rank={r["rank"]} score={r["score"]:.3f} | {r["text"][:60]}...")


## Strategy 1: Reranking

**Idea:** Score each chunk's relevance to the query more carefully,
then keep only the top-N most relevant chunks.

The initial retrieval uses bi-encoder similarity (fast but approximate).
Reranking uses a more precise comparison -- here we compute similarity
between the query and each individual sentence in the chunk, taking
the max. This acts as a simple proxy for cross-encoder reranking.

In production, a **cross-encoder** model (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`)
would be used here for much better precision.


In [ ]:
def rerank_chunks(query: str, chunks: List[Dict], top_n: int = K_RERANK) -> List[Dict]:
    """Rerank chunks by max sentence-level similarity to query.
    
    Instead of chunk-level embedding, we score each sentence in the chunk
    against the query and use the max score. This surfaces chunks with
    at least one highly relevant sentence.
    """
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    
    scored = []
    for chunk in chunks:
        sentences = re.split(r'(?<=[.!?])\s+', chunk["text"])
        sentences = [s for s in sentences if len(s.strip()) > 10]
        if not sentences:
            scored.append((chunk, 0.0))
            continue
        s_embs = encoder.encode(sentences, convert_to_numpy=True)
        s_embs = s_embs / np.linalg.norm(s_embs, axis=1, keepdims=True)
        sims = (s_embs @ q_emb.T).flatten()
        max_sim = float(np.max(sims))
        scored.append((chunk, max_sim))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    return [{"id": c["id"], "text": c["text"], "rerank_score": s, "rank": i + 1}
            for i, (c, s) in enumerate(scored[:top_n])]


# Demo
demo_chunks = retrieve("serverless vs containers differences", k=K_RETRIEVE)
reranked = rerank_chunks("serverless vs containers differences", demo_chunks, top_n=K_RERANK)
print(f"Reranking: {K_RETRIEVE} chunks -> {len(reranked)} chunks")
for r in reranked:
    print(f"  [{r["id"]}] rerank_score={r["rerank_score"]:.3f} | {r["text"][:70]}...")


## Strategy 2: Extractive Compression

**Idea:** Instead of keeping or dropping entire chunks, extract only
the sentences within each chunk that are relevant to the query.

This preserves the original wording (no paraphrasing) while removing
irrelevant sentences. The result is shorter, focused chunks.

**Trade-off:** More aggressive compression than reranking, but may
lose context that helps interpret the relevant sentences.


In [ ]:
def extractive_compress(query: str, chunks: List[Dict],
                        threshold: float = RELEVANCE_THRESHOLD) -> List[Dict]:
    """Extract only query-relevant sentences from each chunk.
    
    For each chunk, keep sentences with cosine similarity >= threshold.
    Drop chunks that have zero relevant sentences.
    """
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    
    compressed = []
    for chunk in chunks:
        sentences = re.split(r'(?<=[.!?])\s+', chunk["text"])
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        if not sentences:
            continue
        s_embs = encoder.encode(sentences, convert_to_numpy=True)
        s_embs = s_embs / np.linalg.norm(s_embs, axis=1, keepdims=True)
        sims = (s_embs @ q_emb.T).flatten()
        
        relevant = [sent for sent, sim in zip(sentences, sims) if sim >= threshold]
        if relevant:
            compressed.append({
                "id": chunk["id"],
                "text": " ".join(relevant),
                "original_sentences": len(sentences),
                "kept_sentences": len(relevant),
            })
    
    return compressed


# Demo
extracted = extractive_compress("serverless vs containers differences", demo_chunks)
print(f"Extractive compression: {K_RETRIEVE} chunks -> {len(extracted)} compressed chunks")
for c in extracted[:5]:
    print(f"  [{c["id"]}] {c["kept_sentences"]}/{c["original_sentences"]} sentences kept")
    print(f"    {c["text"][:100]}...")


## Strategy 3: Abstractive Compression

**Idea:** Instead of selecting sentences, generate a concise summary
of all retrieved chunks focused on the query.

In production, an LLM summarizes the chunks. Here we simulate this
by selecting the single most relevant sentence from each chunk and
combining them into a coherent evidence paragraph.

This achieves maximum compression -- all evidence in one paragraph.

**Trade-off:** Highest compression but loses original wording.
In LLM-based abstractive summary, there's a risk of information
distortion or hallucination during summarization.


In [ ]:
def abstractive_compress(query: str, chunks: List[Dict]) -> str:
    """Create a single evidence paragraph from the most relevant sentences.
    
    Selects the top sentence from each chunk (ranked by query relevance)
    and combines them. This simulates what an LLM summarizer would produce.
    """
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    
    best_sentences = []
    for chunk in chunks:
        sentences = re.split(r'(?<=[.!?])\s+', chunk["text"])
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        if not sentences:
            continue
        s_embs = encoder.encode(sentences, convert_to_numpy=True)
        s_embs = s_embs / np.linalg.norm(s_embs, axis=1, keepdims=True)
        sims = (s_embs @ q_emb.T).flatten()
        best_idx = int(np.argmax(sims))
        best_sentences.append((sentences[best_idx], float(sims[best_idx]), chunk["id"]))
    
    # Sort by relevance, take top sentences
    best_sentences.sort(key=lambda x: x[1], reverse=True)
    top = best_sentences[:6]  # cap at 6 sentences for concise summary
    
    # Deduplicate very similar sentences
    seen = set()
    unique = []
    for sent, score, cid in top:
        key = sent[:50].lower()
        if key not in seen:
            seen.add(key)
            unique.append((sent, score, cid))
    
    summary = " ".join(s for s, _, _ in unique)
    return summary


# Demo
summary = abstractive_compress("serverless vs containers differences", demo_chunks)
print(f"Abstractive compression: {K_RETRIEVE} chunks -> 1 paragraph")
print(f"  Summary ({len(summary)} chars):")
print(f"  {summary[:200]}...")


## Evaluation Framework

We measure three things:

| Metric | What it measures | Formula |
|--------|-----------------|----------|
| **Compression ratio** | How much context was reduced | 1 - compressed_chars / original_chars |
| **Information retention** | How much relevant info survived | cosine_sim(compressed, ideal_answer) |
| **Relevance density** | Fraction of compressed text that's relevant | relevant_sentences / total_sentences |


In [ ]:
@dataclass
class CompressionResult:
    """Result of applying a compression strategy."""
    strategy: str
    question: str
    original_chars: int
    compressed_chars: int
    compression_ratio: float
    information_retention: float  # sim(compressed, ideal_answer_context)
    num_chunks_in: int
    num_chunks_out: int  # or 1 for abstractive
    compressed_text: str


def compute_info_retention(compressed_text: str, reference_texts: List[str]) -> float:
    """Measure how much information from reference chunks is retained.
    
    Computes cosine similarity between compressed text and each reference.
    Returns the average similarity (how well the compressed text covers
    the reference information).
    """
    if not compressed_text or not reference_texts:
        return 0.0
    c_emb = encoder.encode([compressed_text], convert_to_numpy=True)
    c_emb = c_emb / np.linalg.norm(c_emb, axis=1, keepdims=True)
    r_embs = encoder.encode(reference_texts, convert_to_numpy=True)
    r_embs = r_embs / np.linalg.norm(r_embs, axis=1, keepdims=True)
    sims = (c_emb @ r_embs.T).flatten()
    return float(np.mean(sims))


print("Evaluation framework defined.")


## Test Questions

Questions with known relevant chunks so we can measure information retention.


In [ ]:
test_questions = [
    {
        "question": "What are the main differences between serverless and containers?",
        "relevant_chunks": ["C02", "C03", "C04"],
        "type": "comparison",
    },
    {
        "question": "How can I optimize cloud costs? Give specific strategies.",
        "relevant_chunks": ["C05", "C20", "C29"],
        "type": "multi-topic",
    },
    {
        "question": "What are cloud security best practices?",
        "relevant_chunks": ["C07", "C17", "C23"],
        "type": "broad",
    },
    {
        "question": "Explain the CI/CD pipeline stages and deployment strategies.",
        "relevant_chunks": ["C14", "C28"],
        "type": "specific",
    },
    {
        "question": "What disaster recovery strategies are available in the cloud?",
        "relevant_chunks": ["C19"],
        "type": "single-topic",
    },
    {
        "question": "Compare microservices and monolithic architectures in the cloud.",
        "relevant_chunks": ["C09", "C18", "C21"],
        "type": "comparison",
    },
]

print(f"Test questions: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"  Q{i} [{q['type']}]: {q['question'][:60]}...")


## Benchmark: All Compression Strategies

For each question:
1. Retrieve K=10 chunks (baseline: no compression)
2. Rerank and keep top-3
3. Extract relevant sentences
4. Abstractive summary
5. Measure compression ratio and information retention


In [ ]:
all_results = []

for qi, tq in enumerate(test_questions, 1):
    question = tq["question"]
    relevant_ids = tq["relevant_chunks"]
    relevant_texts = [c["text"] for c in corpus if c["id"] in relevant_ids]
    
    # Retrieve K=10
    chunks = retrieve(question, k=K_RETRIEVE)
    original_text = " ".join(c["text"] for c in chunks)
    original_chars = len(original_text)
    
    # --- No compression baseline ---
    retention_none = compute_info_retention(original_text, relevant_texts)
    all_results.append(CompressionResult(
        strategy="None (K=10)", question=question,
        original_chars=original_chars, compressed_chars=original_chars,
        compression_ratio=0.0, information_retention=retention_none,
        num_chunks_in=K_RETRIEVE, num_chunks_out=K_RETRIEVE,
        compressed_text=original_text,
    ))
    
    # --- Reranking ---
    reranked = rerank_chunks(question, chunks, top_n=K_RERANK)
    reranked_text = " ".join(r["text"] for r in reranked)
    retention_rerank = compute_info_retention(reranked_text, relevant_texts)
    all_results.append(CompressionResult(
        strategy="Reranking", question=question,
        original_chars=original_chars, compressed_chars=len(reranked_text),
        compression_ratio=1 - len(reranked_text) / original_chars,
        information_retention=retention_rerank,
        num_chunks_in=K_RETRIEVE, num_chunks_out=len(reranked),
        compressed_text=reranked_text,
    ))
    
    # --- Extractive ---
    extracted = extractive_compress(question, chunks)
    extracted_text = " ".join(e["text"] for e in extracted)
    retention_extract = compute_info_retention(extracted_text, relevant_texts)
    all_results.append(CompressionResult(
        strategy="Extractive", question=question,
        original_chars=original_chars, compressed_chars=len(extracted_text),
        compression_ratio=1 - len(extracted_text) / original_chars,
        information_retention=retention_extract,
        num_chunks_in=K_RETRIEVE, num_chunks_out=len(extracted),
        compressed_text=extracted_text,
    ))
    
    # --- Abstractive ---
    summary = abstractive_compress(question, chunks)
    retention_abstract = compute_info_retention(summary, relevant_texts)
    all_results.append(CompressionResult(
        strategy="Abstractive", question=question,
        original_chars=original_chars, compressed_chars=len(summary),
        compression_ratio=1 - len(summary) / original_chars,
        information_retention=retention_abstract,
        num_chunks_in=K_RETRIEVE, num_chunks_out=1,
        compressed_text=summary,
    ))

print(f"Benchmark complete: {len(test_questions)} questions x 4 strategies = {len(all_results)} results")


## Aggregate Results

In [ ]:
strategies = ["None (K=10)", "Reranking", "Extractive", "Abstractive"]

print(f"{'Strategy':<16} {'Compress%':>10} {'InfoRetain':>10} {'AvgCharsOut':>12} {'ChunksOut':>10}")
print("-" * 62)

for strat in strategies:
    strat_results = [r for r in all_results if r.strategy == strat]
    avg_comp = np.mean([r.compression_ratio for r in strat_results])
    avg_retain = np.mean([r.information_retention for r in strat_results])
    avg_chars = np.mean([r.compressed_chars for r in strat_results])
    avg_chunks = np.mean([r.num_chunks_out for r in strat_results])
    print(f"{strat:<16} {avg_comp:>9.1%} {avg_retain:>10.3f} {avg_chars:>12.0f} {avg_chunks:>10.1f}")

print(f"\nInterpretation:")
print(f"  - Higher compression = less context sent to the generator")
print(f"  - Higher info retention = more relevant information preserved")
print(f"  - Best strategy maximizes retention while maximizing compression")


## Per-Question Breakdown

In [ ]:
for qi, tq in enumerate(test_questions, 1):
    question = tq["question"]
    print(f"\nQ{qi}: {question}")
    print(f"  Type: {tq['type']} | Relevant chunks: {tq['relevant_chunks']}")
    print(f"  {'Strategy':<16} {'Compress%':>10} {'InfoRetain':>10} {'Chars':>8}")
    print(f"  {"-"*46}")
    
    for strat in strategies:
        r = [x for x in all_results if x.strategy == strat and x.question == question][0]
        print(f"  {strat:<16} {r.compression_ratio:>9.1%} {r.information_retention:>10.3f} {r.compressed_chars:>8}")


## Qualitative Comparison

See how the compressed context looks for one example question.


In [ ]:
example_q = test_questions[0]["question"]
print(f"Question: {example_q}\n")

for strat in strategies:
    r = [x for x in all_results if x.strategy == strat and x.question == example_q][0]
    print(f"--- {strat} ---")
    print(f"  Compression: {r.compression_ratio:.1%} | Retention: {r.information_retention:.3f}")
    print(f"  {r.num_chunks_in} -> {r.num_chunks_out} units | {r.original_chars} -> {r.compressed_chars} chars")
    # Show first 200 chars of compressed text
    preview = r.compressed_text[:200].replace("\n", " ")
    print(f"  Preview: {preview}...")
    print()


## Retrieval Quality: Did We Retrieve the Right Chunks?

Compression only works well if the initial retrieval captures the
relevant chunks. Let's check.


In [ ]:
print("Retrieval quality check (K=10)\n")

for qi, tq in enumerate(test_questions, 1):
    question = tq["question"]
    relevant_ids = set(tq["relevant_chunks"])
    chunks = retrieve(question, k=K_RETRIEVE)
    retrieved_ids = set(c["id"] for c in chunks)
    
    found = relevant_ids & retrieved_ids
    missed = relevant_ids - retrieved_ids
    recall = len(found) / len(relevant_ids) if relevant_ids else 0
    
    status = "FULL" if not missed else "PARTIAL"
    print(f"  Q{qi}: recall={recall:.0%} [{status}] | found={sorted(found)} | missed={sorted(missed)}")

print(f"\nNote: If relevant chunks are not in the top-10 retrieval,")
print(f"no compression strategy can recover that information.")


## Error Analysis

When does compression help or hurt information retention?


In [ ]:
print("Error Analysis: When Does Compression Help/Hurt?\n")

for qi, tq in enumerate(test_questions, 1):
    question = tq["question"]
    none_r = [x for x in all_results if x.strategy == "None (K=10)" and x.question == question][0]
    
    for strat in ["Reranking", "Extractive", "Abstractive"]:
        r = [x for x in all_results if x.strategy == strat and x.question == question][0]
        delta = r.information_retention - none_r.information_retention
        if delta < -0.02:  # meaningful loss
            print(f"  Q{qi} [{tq['type']}]: {strat} HURT retention by {delta:+.3f}")
            print(f"    Compression={r.compression_ratio:.1%}, Retention dropped {none_r.information_retention:.3f} -> {r.information_retention:.3f}")
            if strat == "Abstractive":
                print(f"    Likely cause: summary missed key details from relevant chunks")
            elif strat == "Extractive":
                print(f"    Likely cause: relevant sentences scored below threshold")
            else:
                print(f"    Likely cause: relevant chunks ranked below top-{K_RERANK}")
            print()

# Best strategy per question type
print("\nBest strategy per question (excluding no-compression baseline):")
for qi, tq in enumerate(test_questions, 1):
    question = tq["question"]
    best_strat = None
    best_retain = -1
    for strat in ["Reranking", "Extractive", "Abstractive"]:
        r = [x for x in all_results if x.strategy == strat and x.question == question][0]
        if r.information_retention > best_retain:
            best_retain = r.information_retention
            best_strat = strat
    print(f"  Q{qi} [{tq['type']:>12s}]: {best_strat} (retention={best_retain:.3f})")


## Compression Efficiency Analysis

The ideal compression strategy maximizes information retention
while minimizing context size. We can compute an efficiency score.


In [ ]:
print("Compression Efficiency (retention / context_size_ratio)\n")
print(f"{'Strategy':<16} {'Efficiency':>10} {'Compress%':>10} {'InfoRetain':>10}")
print("-" * 48)

for strat in strategies:
    strat_results = [r for r in all_results if r.strategy == strat]
    avg_retain = np.mean([r.information_retention for r in strat_results])
    avg_size_ratio = np.mean([r.compressed_chars / r.original_chars for r in strat_results])
    efficiency = avg_retain / avg_size_ratio if avg_size_ratio > 0 else 0
    avg_comp = np.mean([r.compression_ratio for r in strat_results])
    print(f"{strat:<16} {efficiency:>10.3f} {avg_comp:>9.1%} {avg_retain:>10.3f}")

print(f"\nHigher efficiency = more information per unit of context size")
print(f"Efficiency > 1.0 means the compression kept relevant info while cutting noise")


## Limitations

1. **No real cross-encoder.** Production reranking uses cross-encoder
   models (e.g., `ms-marco-MiniLM`) that score query-document pairs
   jointly. Our sentence-level max-sim is a rough proxy.

2. **No LLM summarization.** Real abstractive compression uses an LLM
   to generate a coherent summary. Our sentence extraction is a
   deterministic approximation.

3. **No generation quality.** We measure information retention in
   embedding space, not actual answer quality from an LLM generator.

4. **Fixed thresholds.** Sentence relevance threshold and top-N
   reranking cutoff should be tuned per domain.

5. **Small corpus.** 30 chunks are manageable; real systems have
   thousands of chunks where compression matters even more.

6. **No latency measurement.** Compression adds compute cost;
   in production, the trade-off is compute vs token savings.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Summarizing before retrieving enough | Low recall means summary misses info | Retrieve wide (K=10+), then compress |
| Using only embeddings for reranking | Bi-encoder misses nuance | Use cross-encoder for reranking |
| Compressing too aggressively | Loses context needed for answer | Monitor information retention |
| Ignoring lost-in-the-middle | LLMs underuse middle context | Put best evidence first/last |
| Same compression for all queries | Different query types need different strategies | Adapt strategy to query complexity |
| No fallback | What if compression removes key info? | Compare answers with and without compression |


## Mini Challenge

1. **Cross-encoder reranking.** Replace sentence-level max-sim with
   `cross-encoder/ms-marco-MiniLM-L-6-v2`. Compare ranking quality.

2. **LLM-based abstractive summary.** Use an LLM to summarize chunks
   instead of selecting top sentences. Compare coherence.

3. **Adaptive K.** Instead of fixed K=10, dynamically decide how many
   chunks to retrieve based on query complexity.

4. **Pipeline with verification.** Combine compression (Project 25)
   with citation verification (Project 24). Verify compressed context
   still supports the generated answer.

5. **Context window optimization.** Given a token budget (e.g., 2000),
   find the compression strategy that maximizes information retention
   within that budget.


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Strategy selection** | Use reranking for simple queries; extractive for factoid; abstractive for synthesis |
| **Latency budget** | Cross-encoder: ~100ms; LLM summary: ~500ms; sentence extraction: ~50ms |
| **Token budget** | Set max context tokens; choose compression level to fit |
| **Caching** | Cache compressed chunks for repeated queries |
| **Monitoring** | Track compression ratio and downstream answer quality |
| **Fallback** | If answer fails quality check, retry with less compression |
| **A/B test** | Compare compressed vs uncompressed answers for user satisfaction |


## How to Improve This Project

1. **Cross-encoder reranking** -- Much better precision than embedding similarity.
2. **LLM-based compression** -- Coherent summaries instead of sentence extraction.
3. **Multi-step compression** -- Rank, then extract, then summarize.
4. **Token budget targeting** -- Compress to exactly N tokens.
5. **Learned compression** -- Train a model to predict which sentences to keep.
6. **Combine with query rewriting** -- Better queries retrieve better chunks,
   making compression more effective.


## Key Takeaways

1. **Retrieve wide, compress tight.** High recall retrieval + intelligent
   compression beats low-K precise retrieval in most cases.

2. **Three compression strategies serve different needs.**
   - Reranking: safest, keeps original chunks intact
   - Extractive: moderate compression, keeps original wording
   - Abstractive: maximum compression, highest information density

3. **Compression ratio vs information retention is the core trade-off.**
   Monitor both; never optimize compression without checking retention.

4. **Lost-in-the-middle is real.** LLMs struggle with information buried
   in long contexts. Compression surfaces the key evidence.

5. **Cross-encoders are worth the latency.** They dramatically improve
   reranking precision compared to bi-encoder similarity.

6. **Composable with the full RAG stack.** Query rewriting (P22) +
   agentic retrieval (P23) + citation verification (P24) + compression
   (P25) = production-grade RAG pipeline.

7. **Always benchmark.** Different query types benefit from different
   compression strategies. One size does not fit all.
